In [ ]:
import os
import torch
import pandas as pd
import sacrebleu
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from fix_punctuation import clean_text


In [ ]:
def plot_loss_curves(histories, save_path="loss_curves.png"):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Training & Validation Loss Curves", fontsize=16, fontweight="bold")

    for ax, (name, hist) in zip(axes.flat, histories.items()):
        epochs = range(1, len(hist["train_losses"]) + 1)
        ax.plot(epochs, hist["train_losses"], label="Train Loss", linewidth=2)
        ax.plot(epochs, hist["val_losses"], label="Val Loss", linewidth=2)
        ax.set_title(name, fontsize=12)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved loss curves to {save_path}")

In [ ]:
def plot_metric_curves(histories, save_path="metric_curves.png"):
    metrics = ["val_bleu", "val_chrf", "val_ter"]
    labels = ["BLEU ↑", "chrF ↑", "TER ↓"]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Dev Set Metrics Over Training", fontsize=16, fontweight="bold")

    for ax, metric, label in zip(axes, metrics, labels):
        for name, hist in histories.items():
            if metric in hist and len(hist[metric]) > 0:
                epochs = range(1, len(hist[metric]) + 1)
                ax.plot(epochs, hist[metric], label=name, linewidth=2)
        ax.set_title(label, fontsize=12)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Score")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved metric curves to {save_path}")

In [ ]:
def plot_model4_tf_overlay(history, save_path="model4_tf_overlay.png"):
    epochs = range(1, len(history["val_losses"]) + 1)

    fig, (ax1, ax3) = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("Model 4: Loss & Teacher Forcing Schedule", fontsize=14, fontweight="bold")

    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Train Loss", color="tab:blue")
    ax1.plot(epochs, history["train_losses"], label="Train Loss", color="tab:blue", linewidth=2)
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax1.legend(loc="upper left")
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.set_ylabel("Teacher Forcing Probability", color="tab:red")
    ax2.plot(epochs, history["tf_probs"], label="TF Prob p(t)", color="tab:red", linewidth=2, linestyle="-.")
    ax2.tick_params(axis="y", labelcolor="tab:red")
    ax2.set_ylim(0, 1)
    ax2.legend(loc="upper right")

    ax3.set_xlabel("Epoch")
    ax3.set_ylabel("Val Loss", color="tab:cyan")
    ax3.plot(epochs, history["val_losses"], label="Val Loss", color="tab:cyan", linewidth=2, linestyle="--")
    ax3.tick_params(axis="y", labelcolor="tab:cyan")
    ax3.legend(loc="upper left")
    ax3.grid(True, alpha=0.3)

    ax4 = ax3.twinx()
    ax4.set_ylabel("Teacher Forcing Probability", color="tab:red")
    ax4.plot(epochs, history["tf_probs"], label="TF Prob p(t)", color="tab:red", linewidth=2, linestyle="-.")
    ax4.tick_params(axis="y", labelcolor="tab:red")
    ax4.set_ylim(0, 1)
    ax4.legend(loc="upper right")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved Model 4 TF overlay to {save_path}")

In [ ]:
def score_predictions(pred_file, ref_csv, tgt_column="translated", label="", save_samples=None):
    with open(pred_file, "r", encoding="utf-8") as f:
        hyps = [clean_text(line.strip()) for line in f]

    df = pd.read_csv(ref_csv)
    refs = [clean_text(str(line)) for line in df[tgt_column]]
    sources = df["source"].tolist() if "source" in df.columns else []

    n = min(len(hyps), len(refs))
    hyps, refs = hyps[:n], refs[:n]

    print(hyps, [refs])
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    chrf = sacrebleu.corpus_chrf(hyps, [refs])
    ter = sacrebleu.corpus_ter(hyps, [refs])

    print(f"\n{'='*50}")
    print(f"  {label or pred_file}")
    print(f"{'='*50}")
    print(f"  BLEU : {bleu.score:.2f}")
    print(f"  chrF : {chrf.score:.2f}")
    print(f"  TER  : {ter.score:.2f}")
    print(f"{'='*50}")

    if save_samples:
        with open(save_samples, "w", encoding="utf-8") as fs:
            fs.write(f"Sample Translations Report: {label}\n")
            fs.write("="*60 + "\n")
            for i in range(min(20, n)):  # save 20 samples
                if sources:
                    fs.write(f"[SRC] English:      {sources[i]}\n")
                fs.write(f"[REF] Ground Truth: {refs[i]}\n")
                fs.write(f"[HYP] Prediction:   {hyps[i]}\n")
                fs.write("-" * 60 + "\n")
        print(f"Saved sample translations to {save_samples}")

    return {"bleu": bleu.score, "chrf": chrf.score, "ter": ter.score}

In [ ]:
if __name__ == "__main__":
    DATA_DIR = "COL775-A1-PART2-Dataset"

    '''Graphs'''
    if os.path.exists("training_histories.pt"):
        histories = torch.load("training_histories.pt", weights_only=False)
        plot_loss_curves(histories)
        plot_metric_curves(histories)
        if "Model4_TunedBERT" in histories:
            plot_model4_tf_overlay(histories["Model4_TunedBERT"])
    else:
        print("[Skip] training_histories.pt not found.")

    '''Score Beam Search Predictions (Model 4)'''
    test_ref = f"{DATA_DIR}/Hindi_public_test.csv"
    for k in [1, 5, 10, 20]:
        pred_file = f"model4_beam_{k}.txt"
        if os.path.exists(pred_file):
            save_dest = "sample_translations.txt" if k == 5 else None
            score_predictions(pred_file, test_ref, label=f"Model 4 — Beam k={k}", save_samples=save_dest)

    '''Score Transfer Learning Predictions'''
    mr_test_ref = f"{DATA_DIR}/Marathi_public_test.csv"
    for pf, lbl in [("transferred_predictions.txt", "Transfer (HI→MR)"),
                     ("baseline_predictions.txt", "Baseline (MR scratch)")]:
        if os.path.exists(pf):
            score_predictions(pf, mr_test_ref, label=lbl)